# SDG 3 Indicator Multi-Label Text Classification
**Group Assignment 2** — Complete pipeline: EDA → Preprocessing → Feature Engineering → Experiments → Evaluation → Inference


### Sections
1. Setup & Installs
2. Load Data
3. Exploratory Data Analysis (EDA)
4. Preprocessing Pipeline
5. Feature Engineering
6. Experiments (8 total)
7. Results Summary & Model Comparison
8. Inference on Test Set


---
## Section 1 — Setup & Installs


In [4]:
# Install required libraries (only needed once per Colab session)
!pip install -q scikit-multilearn imbalanced-learn sentence-transformers seaborn

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# NLP
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import hamming_loss, classification_report, f1_score
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

# scikit-multilearn
from skmultilearn.model_selection import iterative_train_test_split

# Sentence Transformers
from sentence_transformers import SentenceTransformer

# Scipy
from scipy.sparse import hstack, issparse

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


print('All libraries loaded successfully.')

All libraries loaded successfully.


---
## Section 2 — Load Data


In [6]:
# ── Load datasets


TRAIN_PATH = 'Devex_train.csv'
TEST_PATH  = 'Devex_test_questions.csv'
def read_csv_safe(path):
    na_vals = ['', '#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan',
               '1.#IND', '1.#QNAN', '<NA>', 'N/A', 'NA', 'NULL', 'NaN', 'n/a', 'nan', 'null']

    for enc in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
        try:
            return pd.read_csv(path, encoding=enc, na_values=na_vals,
                               keep_default_na=True, engine='python',
                               on_bad_lines='skip')
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f'  [{enc}] failed: {e}')
            continue
    raise ValueError(f'Could not read {path} with any common encoding')

train_df = read_csv_safe(TRAIN_PATH)
test_df  = read_csv_safe(TEST_PATH)

print('Train shape:', train_df.shape)
print('Test  shape:', test_df.shape)
print()
print('Train columns:', train_df.columns.tolist())
print('Test  columns:', test_df.columns.tolist())

Train shape: (2995, 15)
Test  shape: (998, 3)

Train columns: ['Unique ID', 'Type', 'Text', 'Label 1', 'Label 2', 'Label 3', 'Label 4', 'Label 5', 'Label 6', 'Label 7', 'Label 8', 'Label 9', 'Label 10', 'Label 11', 'Label 12']
Test  columns: ['Unique ID', 'Type', 'Text']


In [7]:
# Preview the training data
train_df.head(3)

,Unique ID,Type,Text,Label 1,Label 2,Label 3,Label 4,Label 5,Label 6,Label 7,Label 8,Label 9,Label 10,Label 11,Label 12
0,12555,Grant,Centers of Biomedical Research Excellence (COB...,3.b.2 - Total net official development assista...,3.c.1 - Health worker density and distribution,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,14108,Grant,Research on Regenerative Medicine <h2><strong>...,3.b.2 - Total net official development assista...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,23168,Organization,Catholic Health Association of India (CHAI): <...,3.d.1 - International Health Regulations (IHR)...,3.8.1 - Coverage of essential health services ...,3.8.2 - Proportion of population with large ho...,3.b.3 - Proportion of health facilities that h...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# ── Identify the text column and label columns

TEXT_COL = 'Text'
LABEL_COLS = [f'Label {i}' for i in range(1, 11)]

# Convert identified label columns to binary (1 if present, 0 if missing/empty)
for col in LABEL_COLS:
    # This function will handle explicit NaN, None, empty strings, and common string representations of missing data
    def is_label_present(val):
        # Explicitly check for actual NaN or None
        if pd.isna(val) or val is None:
            return 0
        # If it's a string, check for empty string after strip, or common "missing" strings
        if isinstance(val, str):
            val_lower = val.strip().lower()
            if val_lower == '' or val_lower in ['nan', 'none', 'null', 'n/a', '-']:
                return 0
        # Otherwise, consider it a present label
        return 1
    train_df[col] = train_df[col].apply(is_label_present)

print(f'Text column  : {TEXT_COL}')
print(f'Label columns ({len(LABEL_COLS)}): {LABEL_COLS[:5]} ...')
print()
print('Sample text:')
print(train_df[TEXT_COL].iloc[0][:300])

Text column  : Text
Label columns (10): ['Label 1', 'Label 2', 'Label 3', 'Label 4', 'Label 5'] ...

Sample text:
Centers of Biomedical Research Excellence (COBRE) Phase III - Transitional Centers     <p><strong>Funding Opportunity Description</strong></p>    <p><a name="_Toc258873267"></a>The Institutional Development Award (IDeA) Program endeavors to stimulate research at institutions in states that have not 


In [9]:
# Basic data quality checks
print('Missing values in train:')
print(train_df.isnull().sum())
print()
print('Missing values in test:')
print(test_df.isnull().sum())

# Fill any missing text with empty string
train_df[TEXT_COL] = train_df[TEXT_COL].fillna('')
test_df[TEXT_COL]  = test_df[TEXT_COL].fillna('')

print('\nMissing values handled.')

Missing values in train:
Unique ID       0
Type            0
Text            0
Label 1         0
Label 2         0
Label 3         0
Label 4         0
Label 5         0
Label 6         0
Label 7         0
Label 8         0
Label 9         0
Label 10        0
Label 11     2995
Label 12     2995
dtype: int64

Missing values in test:
Unique ID    0
Type         0
Text         0
dtype: int64

Missing values handled.


---
## Section 6 — Experiments
Each cell is one experiment. Run them in order.

In [10]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)       # remove URLs
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)          # remove special chars
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

train_df['clean_text'] = train_df[TEXT_COL].apply(preprocess_text)
test_df['clean_text']  = test_df[TEXT_COL].apply(preprocess_text)
print('Preprocessing done.')

Preprocessing done.


In [11]:
mlb = MultiLabelBinarizer()

# Build label lists from the 10 label columns
def get_label_list(row):
    return [col for col in LABEL_COLS if row[col] == 1]

train_df['label_list'] = train_df.apply(get_label_list, axis=1)
Y_all = mlb.fit_transform(train_df['label_list'])
print('Labels shape:', Y_all.shape)
print('Classes:', mlb.classes_)

Labels shape: (2995, 10)
Classes: ['Label 1' 'Label 10' 'Label 2' 'Label 3' 'Label 4' 'Label 5' 'Label 6'
 'Label 7' 'Label 8' 'Label 9']


In [12]:
X_text = train_df['clean_text'].values

X_text_train, X_text_val, Y_train, Y_val = train_test_split(
    X_text, Y_all, test_size=0.2, random_state=RANDOM_STATE
)
print('Train size:', len(X_text_train), '| Val size:', len(X_text_val))

Train size: 2396 | Val size: 599


In [13]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_text_train)
X_val_tfidf   = tfidf.transform(X_text_val)
print('TF-IDF train shape:', X_train_tfidf.shape)

TF-IDF train shape: (2396, 20000)


In [14]:
results = []

def evaluate(y_true, y_pred, name):
    hl  = hamming_loss(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1w = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(f'{name}')
    print(f'  Hamming Loss : {hl:.4f}')
    print(f'  F1 Macro     : {f1m:.4f}')
    print(f'  F1 Weighted  : {f1w:.4f}\n')
    return {'Experiment': name, 'Hamming Loss': hl, 'F1 Macro': f1m, 'F1 Weighted': f1w}

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 1 — TF-IDF + Logistic Regression (BASELINE)
# Why: Establish a simple sparse-feature baseline.
# ══════════════════════════════════════════════════════════════════════════════
clf_exp1 = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE),
    n_jobs=-1
)
clf_exp1.fit(X_train_tfidf, Y_train)
pred_exp1 = clf_exp1.predict(X_val_tfidf)

res = evaluate(Y_val, pred_exp1, 'Exp 1: TF-IDF + Logistic Regression')
results.append(res)

Exp 1: TF-IDF + Logistic Regression
  Hamming Loss : 0.0775
  F1 Macro     : 0.1648
  F1 Weighted  : 0.6813



In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 2 — TF-IDF + Random Forest
# Why: Test a tree-based ensemble on the same sparse features as Exp 1.
#      Does non-linearity help over logistic regression?
# ══════════════════════════════════════════════════════════════════════════════
clf_exp2 = OneVsRestClassifier(
    RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
)
clf_exp2.fit(X_train_tfidf, Y_train)
pred_exp2 = clf_exp2.predict(X_val_tfidf)

res = evaluate(Y_val, pred_exp2, 'Exp 2: TF-IDF + Random Forest')
results.append(res)

Exp 2: TF-IDF + Random Forest
  Hamming Loss : 0.0741
  F1 Macro     : 0.2309
  F1 Weighted  : 0.7549



In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT 3 — TF-IDF + LinearSVC
# Why: LinearSVC is often strong on high-dimensional sparse text. Compare to LR.
# ══════════════════════════════════════════════════════════════════════════════
clf_exp3 = OneVsRestClassifier(
    LinearSVC(max_iter=2000, C=0.5, random_state=RANDOM_STATE),
    n_jobs=-1
)
clf_exp3.fit(X_train_tfidf, Y_train)
pred_exp3 = clf_exp3.predict(X_val_tfidf)

res = evaluate(Y_val, pred_exp3, 'Exp 3: TF-IDF + LinearSVC')
results.append(res)

Exp 3: TF-IDF + LinearSVC
  Hamming Loss : 0.0810
  F1 Macro     : 0.1556
  F1 Weighted  : 0.6497

